In [1]:
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans
from dataclasses import dataclass, field

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _get(d: Dict[str, Any], k: str, default=None):
    return d.get(k, default)


@dataclass
class NERMetricsCollector:
    overall_rows: List[Dict[str, Any]] = field(default_factory=list)
    label_rows: List[Dict[str, Any]] = field(default_factory=list)
    meta: Dict[str, Any] = field(
        default_factory=dict
    )  # ex.: nome do modelo, dataset, etc.

    def record(
        self,
        split_name: Any,
        metrics: Dict[str, Any],
        extras: Optional[Dict[str, Any]] = None,
    ):
        """
        Registra os resultados de um split.
        - split_name: pode ser string, int, tupla... será convertido para string
        - metrics: dict retornado pelo seu train/eval
        - extras: (opcional) dict com metadados (seed, versão, etc.)
        """
        split_str = str(split_name)

        # ------------------------------
        # Tabela 1: métricas gerais
        # ------------------------------
        overall = {
            "split": split_str,
            "eval_loss": _get(metrics, "eval_loss"),
            "overall_precision": _get(metrics, "eval_overall_precision"),
            "overall_recall": _get(metrics, "eval_overall_recall"),
            "overall_f1": _get(metrics, "eval_overall_f1"),
            "overall_accuracy": _get(metrics, "eval_overall_accuracy"),
            "f1_micro": _get(metrics, "eval_f1_micro"),
            "f1_macro": _get(metrics, "eval_f1_macro"),
            "f1_weighted": _get(metrics, "eval_f1_weighted"),
            "runtime_s": _get(metrics, "eval_runtime"),
            "samples_per_sec": _get(metrics, "eval_samples_per_second"),
            "steps_per_sec": _get(metrics, "eval_steps_per_second"),
            "epoch": _get(metrics, "epoch"),
        }

        self.overall_rows.append(overall)

        # ------------------------------
        # Tabela 2: métricas por rótulo
        # ------------------------------
        # Regra: qualquer entrada do dict que seja outro dict contendo
        # precision/recall/f1/number é tratada como rótulo.
        for k, v in metrics.items():
            if isinstance(v, dict) and {"precision", "recall", "f1", "number"} <= set(
                v.keys()
            ):
                self.label_rows.append(
                    {
                        "split": split_str,
                        "label": k.replace(
                            "eval_", ""
                        ),  # remove prefixo "eval_" para ficar limpo
                        "precision": v["precision"],
                        "recall": v["recall"],
                        "f1": v["f1"],
                        "support": v["number"],
                    }
                )

    # Comentário: retorna DataFrames prontos para inspeção ou export
    def to_dataframes(self):
        df_overall = pd.DataFrame(self.overall_rows)
        df_labels = pd.DataFrame(self.label_rows)
        return df_overall, df_labels

    # Comentário: exporta dois CSVs (UTF-8 com BOM para abrir liso no Excel)
    def to_csv(self, base_name: str = "ner"):
        df_overall, df_labels = self.to_dataframes()
        df_overall.to_csv(
            f"{base_name}_overall_metrics.csv", index=False, encoding="utf-8-sig"
        )
        df_labels.to_csv(
            f"{base_name}_label_metrics.csv", index=False, encoding="utf-8-sig"
        )
        return f"{base_name}_overall_metrics.csv", f"{base_name}_label_metrics.csv"


# --- Cria (ou reaproveita) um coletor global ---
if "ner_collector" not in globals():
    ner_collector = NERMetricsCollector()

# Configuração e Verificação Inicial

In [3]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

MODEL_NAME = "albert/albert-base-v2"

In [4]:
def read_conll(path):
    tokens, tags = [], []
    sent_tokens, sent_tags = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()
            if not line:
                if tokens:
                    sent_tokens.append(tokens)
                    sent_tags.append(tags)
                    tokens, tags = [], []
            else:
                tok, _, _, ner = line.split()
                tokens.append(tok)
                tags.append(ner)
    # último sentença
    if tokens:
        sent_tokens.append(tokens)
        sent_tags.append(tags)
    return sent_tokens, sent_tags

In [5]:
def read_conll(path: Path, start_sentence_id: int = 0):
    """
    Lê arquivos CoNLL/ CleanCoNLL:
      • usa parts[0] como token
      • usa parts[-1] como rótulo NER (corrigido)
      • ignora linhas '-DOCSTART- …'
    """
    tokens, tags, sent_ids = [], [], []
    cur_toks, cur_tags = [], []
    sid = start_sentence_id

    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()

            if not line:  # fim da sentença
                if cur_toks:
                    tokens.append(cur_toks)
                    tags.append(cur_tags)
                    sent_ids.append(sid)
                    sid += 1
                    cur_toks, cur_tags = [], []
                continue

            parts = line.split()
            if parts[0] == "-DOCSTART-":  # pula marcador de doc
                continue

            tok, ner = parts[0], parts[-1]  # 1ª e última coluna
            cur_toks.append(tok)
            cur_tags.append(ner)

    if cur_toks:  # última sentença
        tokens.append(cur_toks)
        tags.append(cur_tags)
        sent_ids.append(sid)

    return tokens, tags, sent_ids, sid  # devolve sid para continuar contagem

In [6]:
def load_cleanconll(base_dir=None, keep_sentence_id=True):
    # 1. Resolva o diretório da forma mais robusta possível
    if base_dir is None:
        # base_dir = Path.home() / "Documents" / "mestrado" / "ner_splits" / "data" -> Windows
        base_dir = Path.home() / "Estudos" / "mestrado" / "data"  # -> Linux
    else:
        base_dir = Path(base_dir).expanduser()

    FILES = {
        "train": "cleanconll.train",
        "dev"  : "cleanconll.dev",
        "test" : "cleanconll.test",
    }

    # 2. Verifique se todos os arquivos existem antes de começar
    missing = [fname for fname in FILES.values() if not (base_dir / fname).exists()]
    if missing:
        raise FileNotFoundError(f"Arquivos não encontrados em {base_dir}: {', '.join(missing)}")

    splits, sid = {}, 0
    for split, fname in FILES.items():
        toks, labs, sids, sid = read_conll(base_dir / fname, sid)
        data = {"tokens": toks, "ner_tags": labs}
        if keep_sentence_id:
            data["sentence_id"] = sids
        splits[split] = Dataset.from_dict(data)

    return DatasetDict(splits)

In [7]:
cleanconll_ds = load_cleanconll()  # pronto!
print(cleanconll_ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 13957
    })
    dev: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 3233
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 3427
    })
})


In [8]:
cleanconll_ds["train"] = cleanconll_ds["train"].add_column(
    "split", ["train"] * len(cleanconll_ds["train"])
)
cleanconll_ds["dev"] = cleanconll_ds["dev"].add_column(
    "split", ["dev"] * len(cleanconll_ds["dev"])
)
cleanconll_ds["test"] = cleanconll_ds["test"].add_column(
    "split", ["test"] * len(cleanconll_ds["test"])
)

cleanconll_full = concatenate_datasets(
    [
        cleanconll_ds["train"],
        cleanconll_ds["dev"],
        cleanconll_ds["test"],
    ]
)

In [9]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in cleanconll_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [10]:
id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'B-PER',
 4: 'I-LOC',
 5: 'I-MISC',
 6: 'I-ORG',
 7: 'I-PER',
 8: 'O'}

In [11]:
NUM_LABELS

9

# Splits

In [12]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [13]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [14]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [16]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [17]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def standard_split_conll(
    dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
):

    ds = DatasetDict(
        {("val" if k == "dev" else k): v for k, v in cleanconll_ds.items()}
    )
    return ds

In [20]:
# standard_split = standard_split_conll(cleanconll_full)
# print('std')
# # random_splt = random_splits(cleanconll_full)
# # print('random')
# heur_len = heur_len_split(cleanconll_full)
# print("heur_len")
# heur_rare = heur_rare_split(cleanconll_full)
# print("heur_rare")
# advers = adversarial_split(cleanconll_full)
# print("advs")
# loc = loc_split(cleanconll_full)
# print("loc")
# semantic = semantic_cluster_split(cleanconll_full)
# print("semantic")
# reverse = reverse_curriculum_split(cleanconll_full)
# print("reverse")

# Experimentos

In [21]:
from sklearn.metrics import f1_score as skl_f1

In [22]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc": loc_split,
            "reverse": reverse_curriculum_split,
            "semantic": semantic_cluster_split,
            "heur_len": heur_len_split,
            "heur_rare": heur_rare_split,
            "std": standard_split_conll,
            "advs": adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []  # p/ seqeval
        flat_preds, flat_labels = [], []  # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro = skl_f1(flat_labels, flat_preds, average="micro", zero_division=0)
        f1_macro = skl_f1(flat_labels, flat_preds, average="macro", zero_division=0)
        f1_weighted = skl_f1(
            flat_labels, flat_preds, average="weighted", zero_division=0
        )

        return {
            **seqeval_metrics,  # overall_precision / recall / f1
            "f1_micro": f1_micro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }

    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        # output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="no",
        # load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none",
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [23]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "std", "advs"]

In [24]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


/tmp/ipykernel_442257/4053031328.py:28: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()
Map: 100%|██████████| 4123/4123 [00:00<00:00, 17906.93 examples/s]
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
/tmp/ipykernel_442257/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.168800,0.053824,"{'precision': 0.9362992922143579, 'recall': 0.9213930348258706, 'f1': 0.9287863590772316, 'number': 1005}","{'precision': 0.8671586715867159, 'recall': 0.8216783216783217, 'f1': 0.8438061041292639, 'number': 572}","{'precision': 0.8477011494252874, 'recall': 0.9209157127991675, 'f1': 0.8827930174563592, 'number': 961}","{'precision': 0.9692044482463644, 'recall': 0.9700342465753424, 'f1': 0.9696191698759093, 'number': 1168}",0.911859,0.921209,0.916510,0.985273,0.985273,0.910005,0.985251
2,0.015900,0.051961,"{'precision': 0.9435084241823588, 'recall': 0.9472636815920398, 'f1': 0.945382323733863, 'number': 1005}","{'precision': 0.8740740740740741, 'recall': 0.8251748251748252, 'f1': 0.8489208633093527, 'number': 572}","{'precision': 0.8688845401174168, 'recall': 0.9240374609781478, 'f1': 0.8956127080181543, 'number': 961}","{'precision': 0.9627434377646062, 'recall': 0.973458904109589, 'f1': 0.9680715197956576, 'number': 1168}",0.919243,0.930653,0.924913,0.986611,0.986611,0.915027,0.986563
3,0.007000,0.060310,"{'precision': 0.9527638190954774, 'recall': 0.9432835820895522, 'f1': 0.9480000000000001, 'number': 1005}","{'precision': 0.8761726078799249, 'recall': 0.8164335664335665, 'f1': 0.8452488687782805, 'number': 572}","{'precision': 0.8831822759315207, 'recall': 0.9125910509885536, 'f1': 0.8976458546571138, 'number': 961}","{'precision': 0.961376994122586, 'recall': 0.9803082191780822, 'f1': 0.9707503179313267, 'number': 1168}",0.925916,0.927415,0.926665,0.986775,0.986775,0.913383,0.986682
4,0.002900,0.060506,"{'precision': 0.9386562804284323, 'recall': 0.9592039800995025, 'f1': 0.9488188976377953, 'number': 1005}","{'precision': 0.8641114982578397, 'recall': 0.8671328671328671, 'f1': 0.8656195462478184, 'number': 572}","{'precision': 0.9039836567926456, 'recall': 0.9209157127991675, 'f1': 0.9123711340206186, 'number': 961}","{'precision': 0.9709401709401709, 'recall': 0.9726027397260274, 'f1': 0.971770744225834, 'number': 1168}",0.928267,0.939288,0.933745,0.988081,0.988081,0.924782,0.988076
5,0.001100,0.062832,"{'precision': 0.951195219123506, 'recall': 0.9502487562189055, 'f1': 0.9507217521154804, 'number': 1005}","{'precision': 0.8705673758865248, 'recall': 0.8583916083916084, 'f1': 0.8644366197183099, 'number': 572}","{'precision': 0.891566265060241, 'recall': 0.9240374609781478, 'f1': 0.9075114971895758, 'number': 961}","{'precision': 0.9726729291204099, 'recall': 0.9751712328767124, 'f1': 0.9739204788371099, 'number': 1168}",0.929853,0.937129,0.933477,0.988048,0.988048,0.926202,0.988025


F1 Macro: 0.9330494000180107
F1 Micro: 0.9907281604132762
F1 Weighted: 0.9907164560738976
{'eval_loss': 0.054959286004304886, 'eval_LOC': {'precision': 0.9679054054054054, 'recall': 0.9491993373826615, 'f1': 0.9584611095623083, 'number': 1811}, 'eval_MISC': {'precision': 0.8671655753040225, 'recall': 0.8720602069614299, 'f1': 0.8696060037523452, 'number': 1063}, 'eval_ORG': {'precision': 0.8910569105691057, 'recall': 0.9298642533936652, 'f1': 0.9100470523110988, 'number': 1768}, 'eval_PER': {'precision': 0.9810473815461347, 'recall': 0.9732805541810985, 'f1': 0.9771485345255837, 'number': 2021}, 'eval_overall_precision': 0.9345780433159074, 'eval_overall_recall': 0.9390664865676122, 'eval_overall_f1': 0.9368168887558017, 'eval_overall_accuracy': 0.9907281604132762, 'eval_f1_micro': 0.9907281604132762, 'eval_f1_macro': 0.9330494000180107, 'eval_f1_weighted': 0.9907164560738976, 'eval_runtime': 48.3899, 'eval_samples_per_second': 85.204, 'eval_steps_per_second': 5.332, 'epoch': 5.0}




0

In [25]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [26]:
!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [27]:
import time

In [28]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


In [ ]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Map: 100%|██████████| 4123/4123 [00:00<00:00, 14994.75 examples/s]
/tmp/ipykernel_442257/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
   # limpa buffers de IP
 

In [ ]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


Map: 100%|██████████| 4180/4180 [00:00<00:00, 31432.88 examples/s]
/tmp/ipykernel_7866/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.164500,0.055008,"{'precision': 0.9476135040745053, 'recall': 0.9465116279069767, 'f1': 0.9470622454915648, 'number': 860}","{'precision': 0.8229508196721311, 'recall': 0.8352745424292846, 'f1': 0.8290668868703551, 'number': 601}","{'precision': 0.9501289767841788, 'recall': 0.9452523524379812, 'f1': 0.9476843910806174, 'number': 1169}","{'precision': 0.9660766961651918, 'recall': 0.9834834834834835, 'f1': 0.9747023809523809, 'number': 666}",0.929305,0.933252,0.931275,0.987243,0.987243,0.933616,0.987056
2,0.017700,0.053602,"{'precision': 0.9587264150943396, 'recall': 0.9453488372093023, 'f1': 0.9519906323185011, 'number': 860}","{'precision': 0.8629690048939641, 'recall': 0.8801996672212978, 'f1': 0.871499176276771, 'number': 601}","{'precision': 0.9538461538461539, 'recall': 0.9546621043627032, 'f1': 0.9542539546814879, 'number': 1169}","{'precision': 0.9559471365638766, 'recall': 0.9774774774774775, 'f1': 0.9665924276169264, 'number': 666}",0.938708,0.943265,0.940981,0.988343,0.988343,0.940593,0.988360
3,0.007300,0.053978,"{'precision': 0.9536500579374276, 'recall': 0.9569767441860465, 'f1': 0.955310504933256, 'number': 860}","{'precision': 0.8830128205128205, 'recall': 0.9168053244592346, 'f1': 0.8995918367346939, 'number': 601}","{'precision': 0.9597602739726028, 'recall': 0.9589392643284859, 'f1': 0.959349593495935, 'number': 1169}","{'precision': 0.9551374819102749, 'recall': 0.990990990990991, 'f1': 0.9727339719970522, 'number': 666}",0.942917,0.957221,0.950015,0.989974,0.989974,0.952482,0.989967
4,0.003000,0.058884,"{'precision': 0.9505747126436782, 'recall': 0.9616279069767442, 'f1': 0.9560693641618498, 'number': 860}","{'precision': 0.922945205479452, 'recall': 0.8968386023294509, 'f1': 0.9097046413502109, 'number': 601}","{'precision': 0.959656652360515, 'recall': 0.9563729683490163, 'f1': 0.958011996572408, 'number': 1169}","{'precision': 0.9688888888888889, 'recall': 0.9819819819819819, 'f1': 0.9753914988814317, 'number': 666}",0.952641,0.952063,0.952352,0.990259,0.990259,0.952177,0.990223
5,0.001600,0.061471,"{'precision': 0.9505747126436782, 'recall': 0.9616279069767442, 'f1': 0.9560693641618498, 'number': 860}","{'precision': 0.9034941763727121, 'recall': 0.9034941763727121, 'f1': 0.9034941763727121, 'number': 601}","{'precision': 0.9637931034482758, 'recall': 0.9563729683490163, 'f1': 0.9600686990124517, 'number': 1169}","{'precision': 0.9718518518518519, 'recall': 0.984984984984985, 'f1': 0.9783743475018644, 'number': 666}",0.950998,0.953883,0.952439,0.990544,0.990544,0.954403,0.990547


F1 Macro: 0.9540100531973619
F1 Micro: 0.9868720594930945
F1 Weighted: 0.9865837850195649
{'eval_loss': 0.08238779753446579, 'eval_LOC': {'precision': 0.9675028506271379, 'recall': 0.9866279069767442, 'f1': 0.9769717904432931, 'number': 1720}, 'eval_MISC': {'precision': 0.8459343794579173, 'recall': 0.8201936376210235, 'f1': 0.8328651685393258, 'number': 723}, 'eval_ORG': {'precision': 0.979539641943734, 'recall': 0.9730691056910569, 'f1': 0.9762936528167219, 'number': 1968}, 'eval_PER': {'precision': 0.9283582089552239, 'recall': 0.9688473520249221, 'f1': 0.9481707317073171, 'number': 321}, 'eval_overall_precision': 0.9517386722866175, 'eval_overall_recall': 0.9543533389687235, 'eval_overall_f1': 0.9530442123034716, 'eval_overall_accuracy': 0.9868720594930945, 'eval_f1_micro': 0.9868720594930945, 'eval_f1_macro': 0.9540100531973619, 'eval_f1_weighted': 0.9865837850195649, 'eval_runtime': 29.4755, 'eval_samples_per_second': 141.813, 'eval_steps_per_second': 8.889, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
   # limpa buffers de IPC


In [ ]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Map: 100%|██████████| 4123/4123 [00:00<00:00, 11957.78 examples/s]
/tmp/ipykernel_7866/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.173600,0.022327,"{'precision': 0.9724770642201835, 'recall': 0.9691428571428572, 'f1': 0.9708070978820836, 'number': 875}","{'precision': 0.9141221374045801, 'recall': 0.9106463878326996, 'f1': 0.9123809523809524, 'number': 526}","{'precision': 0.9527272727272728, 'recall': 0.9730733519034355, 'f1': 0.9627928341754708, 'number': 1077}","{'precision': 0.9894636015325671, 'recall': 0.9894636015325671, 'f1': 0.9894636015325671, 'number': 1044}",0.962712,0.967632,0.965166,0.994485,0.994485,0.969648,0.994483
2,0.017200,0.020092,"{'precision': 0.9894613583138173, 'recall': 0.9657142857142857, 'f1': 0.9774436090225564, 'number': 875}","{'precision': 0.8957952468007313, 'recall': 0.9315589353612167, 'f1': 0.913327120223672, 'number': 526}","{'precision': 0.9657089898053753, 'recall': 0.9675023212627669, 'f1': 0.9666048237476808, 'number': 1077}","{'precision': 0.9867046533713201, 'recall': 0.9952107279693486, 'f1': 0.9909394372913686, 'number': 1044}",0.966884,0.969903,0.968391,0.995124,0.995124,0.974298,0.995143
3,0.008400,0.021446,"{'precision': 0.977116704805492, 'recall': 0.976, 'f1': 0.9765580331618068, 'number': 875}","{'precision': 0.9209558823529411, 'recall': 0.9524714828897338, 'f1': 0.936448598130841, 'number': 526}","{'precision': 0.972922502334267, 'recall': 0.9675023212627669, 'f1': 0.9702048417132216, 'number': 1077}","{'precision': 0.9875954198473282, 'recall': 0.9913793103448276, 'f1': 0.9894837476099427, 'number': 1044}",0.970314,0.974446,0.972376,0.995225,0.995225,0.975644,0.995242
4,0.002400,0.021191,"{'precision': 0.9817142857142858, 'recall': 0.9817142857142858, 'f1': 0.9817142857142858, 'number': 875}","{'precision': 0.9384615384615385, 'recall': 0.9277566539923955, 'f1': 0.9330783938814531, 'number': 526}","{'precision': 0.9686635944700461, 'recall': 0.9758588672237697, 'f1': 0.9722479185938946, 'number': 1077}","{'precision': 0.9895238095238095, 'recall': 0.9952107279693486, 'f1': 0.9923591212989494, 'number': 1044}",0.973654,0.975866,0.974759,0.995628,0.995628,0.978798,0.995627
5,0.001200,0.021611,"{'precision': 0.9851428571428571, 'recall': 0.9851428571428571, 'f1': 0.9851428571428571, 'number': 875}","{'precision': 0.935361216730038, 'recall': 0.935361216730038, 'f1': 0.935361216730038, 'number': 526}","{'precision': 0.9703703703703703, 'recall': 0.9730733519034355, 'f1': 0.9717199814557255, 'number': 1077}","{'precision': 0.9885496183206107, 'recall': 0.9923371647509579, 'f1': 0.9904397705544934, 'number': 1044}",0.974214,0.976150,0.975181,0.995830,0.995830,0.979970,0.995830


F1 Macro: 0.962138507306871
F1 Micro: 0.9929633431570581
F1 Weighted: 0.9929258121728596
{'eval_loss': 0.03630712628364563, 'eval_LOC': {'precision': 0.967979002624672, 'recall': 0.9761778718898888, 'f1': 0.9720611491829204, 'number': 1889}, 'eval_MISC': {'precision': 0.9012567324955116, 'recall': 0.9061371841155235, 'f1': 0.9036903690369037, 'number': 1108}, 'eval_ORG': {'precision': 0.9612612612612612, 'recall': 0.9560931899641577, 'f1': 0.9586702605570531, 'number': 2232}, 'eval_PER': {'precision': 0.9855289421157685, 'recall': 0.9894789579158316, 'f1': 0.9875, 'number': 1996}, 'eval_overall_precision': 0.9605135993372912, 'eval_overall_recall': 0.9629065743944637, 'eval_overall_f1': 0.9617085982858723, 'eval_overall_accuracy': 0.9929633431570581, 'eval_f1_micro': 0.9929633431570581, 'eval_f1_macro': 0.962138507306871, 'eval_f1_weighted': 0.9929258121728596, 'eval_runtime': 62.6548, 'eval_samples_per_second': 65.805, 'eval_steps_per_second': 4.118, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
   # limpa buffers de IPC

 

In [ ]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Map: 100%|██████████| 4123/4123 [00:00<00:00, 14054.56 examples/s]
/tmp/ipykernel_7866/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.172500,0.032165,"{'precision': 0.9757442116868799, 'recall': 0.9424920127795527, 'f1': 0.9588299024918743, 'number': 939}","{'precision': 0.8382352941176471, 'recall': 0.9011857707509882, 'f1': 0.8685714285714287, 'number': 506}","{'precision': 0.9378330373001776, 'recall': 0.9635036496350365, 'f1': 0.9504950495049505, 'number': 1096}","{'precision': 0.9831460674157303, 'recall': 0.9864712514092446, 'f1': 0.9848058525604952, 'number': 887}",0.943755,0.954492,0.949094,0.991053,0.991053,0.952119,0.990992
2,0.015200,0.029311,"{'precision': 0.9731471535982814, 'recall': 0.9648562300319489, 'f1': 0.9689839572192512, 'number': 939}","{'precision': 0.8745387453874539, 'recall': 0.9367588932806324, 'f1': 0.9045801526717557, 'number': 506}","{'precision': 0.9733944954128441, 'recall': 0.968065693430657, 'f1': 0.9707227813357732, 'number': 1096}","{'precision': 0.989841986455982, 'recall': 0.9887260428410372, 'f1': 0.9892836999435984, 'number': 887}",0.962018,0.967911,0.964956,0.993190,0.993190,0.961289,0.993234
3,0.007000,0.029752,"{'precision': 0.9733759318423855, 'recall': 0.9733759318423855, 'f1': 0.9733759318423855, 'number': 939}","{'precision': 0.9206349206349206, 'recall': 0.9169960474308301, 'f1': 0.9188118811881189, 'number': 506}","{'precision': 0.9761904761904762, 'recall': 0.9726277372262774, 'f1': 0.9744058500914077, 'number': 1096}","{'precision': 0.9875846501128668, 'recall': 0.9864712514092446, 'f1': 0.9870276367738295, 'number': 887}",0.970184,0.968203,0.969193,0.993951,0.993951,0.965345,0.993920
4,0.003100,0.032532,"{'precision': 0.9701810436634718, 'recall': 0.9701810436634718, 'f1': 0.9701810436634718, 'number': 939}","{'precision': 0.9288537549407114, 'recall': 0.9288537549407114, 'f1': 0.9288537549407114, 'number': 506}","{'precision': 0.9726775956284153, 'recall': 0.9744525547445255, 'f1': 0.9735642661804923, 'number': 1096}","{'precision': 0.9887260428410372, 'recall': 0.9887260428410372, 'f1': 0.9887260428410372, 'number': 887}",0.969679,0.970245,0.969962,0.994241,0.994241,0.967883,0.994234
5,0.001100,0.032169,"{'precision': 0.9712460063897763, 'recall': 0.9712460063897763, 'f1': 0.9712460063897763, 'number': 939}","{'precision': 0.9325396825396826, 'recall': 0.9288537549407114, 'f1': 0.9306930693069307, 'number': 506}","{'precision': 0.9726277372262774, 'recall': 0.9726277372262774, 'f1': 0.9726277372262774, 'number': 1096}","{'precision': 0.9887387387387387, 'recall': 0.9898534385569335, 'f1': 0.9892957746478874, 'number': 887}",0.970528,0.970245,0.970387,0.994277,0.994277,0.967052,0.994266


F1 Macro: 0.9478290385962403
F1 Micro: 0.9910221927465213
F1 Weighted: 0.9909641735366185
{'eval_loss': 0.05344154313206673, 'eval_LOC': {'precision': 0.9645580978017048, 'recall': 0.9623992837958818, 'f1': 0.9634774815146762, 'number': 2234}, 'eval_MISC': {'precision': 0.8969072164948454, 'recall': 0.8907849829351536, 'f1': 0.8938356164383562, 'number': 1465}, 'eval_ORG': {'precision': 0.909692671394799, 'recall': 0.9267822736030829, 'f1': 0.9181579575280362, 'number': 2076}, 'eval_PER': {'precision': 0.9795493934142114, 'recall': 0.983983286908078, 'f1': 0.9817613340281397, 'number': 2872}, 'eval_overall_precision': 0.9448410870566559, 'eval_overall_recall': 0.9488840060136463, 'eval_overall_f1': 0.9468582309157002, 'eval_overall_accuracy': 0.9910221927465213, 'eval_f1_micro': 0.9910221927465213, 'eval_f1_macro': 0.9478290385962403, 'eval_f1_weighted': 0.9909641735366185, 'eval_runtime': 63.549, 'eval_samples_per_second': 64.879, 'eval_steps_per_second': 4.06, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
   # limpa buffers de IPC


In [ ]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: std


Map: 100%|██████████| 3427/3427 [00:00<00:00, 21843.61 examples/s]
/tmp/ipykernel_7866/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.164000,0.041421,"{'precision': 0.9327837666455295, 'recall': 0.960809928151535, 'f1': 0.9465894465894468, 'number': 1531}","{'precision': 0.849146110056926, 'recall': 0.9104781281790437, 'f1': 0.8787432498772705, 'number': 983}","{'precision': 0.9293966623876765, 'recall': 0.8921749845964264, 'f1': 0.9104055328513047, 'number': 1623}","{'precision': 0.9668658337859859, 'recall': 0.9732094040459267, 'f1': 0.9700272479564033, 'number': 1829}",0.927695,0.937647,0.932644,0.989335,0.989335,0.936923,0.989380
2,0.014700,0.045679,"{'precision': 0.944015444015444, 'recall': 0.9581972566949706, 'f1': 0.9510534846029173, 'number': 1531}","{'precision': 0.8392523364485981, 'recall': 0.91353001017294, 'f1': 0.8748173404773502, 'number': 983}","{'precision': 0.9381443298969072, 'recall': 0.8971041281577325, 'f1': 0.9171653543307087, 'number': 1623}","{'precision': 0.9654427645788337, 'recall': 0.9775833788955713, 'f1': 0.9714751426242869, 'number': 1829}",0.930491,0.940161,0.935301,0.988848,0.988848,0.933645,0.988936
3,0.007100,0.046012,"{'precision': 0.9490651192778853, 'recall': 0.961463096015676, 'f1': 0.9552238805970149, 'number': 1531}","{'precision': 0.8677766895200784, 'recall': 0.901322482197355, 'f1': 0.8842315369261476, 'number': 983}","{'precision': 0.9290161892901619, 'recall': 0.9192852741836106, 'f1': 0.9241251161350263, 'number': 1623}","{'precision': 0.9747530186608123, 'recall': 0.9710224166211044, 'f1': 0.9728841413311422, 'number': 1829}",0.937667,0.943010,0.940331,0.990172,0.990172,0.942564,0.990158
4,0.003600,0.048619,"{'precision': 0.9506093649775497, 'recall': 0.9679947746570868, 'f1': 0.9592233009708737, 'number': 1531}","{'precision': 0.8825271470878578, 'recall': 0.9094608341810784, 'f1': 0.8957915831663327, 'number': 983}","{'precision': 0.9367167919799498, 'recall': 0.9211337030191005, 'f1': 0.9288598943771358, 'number': 1623}","{'precision': 0.9731800766283525, 'recall': 0.9721159103335156, 'f1': 0.9726477024070023, 'number': 1829}",0.942285,0.946866,0.944570,0.990834,0.990834,0.948880,0.990826
5,0.001100,0.051857,"{'precision': 0.9487179487179487, 'recall': 0.9666884389288047, 'f1': 0.9576188935619542, 'number': 1531}","{'precision': 0.8852621167161226, 'recall': 0.9104781281790437, 'f1': 0.8976930792377131, 'number': 983}","{'precision': 0.9371464487743557, 'recall': 0.9186691312384473, 'f1': 0.9278158058494089, 'number': 1623}","{'precision': 0.9763995609220637, 'recall': 0.9726626571897211, 'f1': 0.9745275267050124, 'number': 1829}",0.943349,0.946195,0.944770,0.990756,0.990756,0.947456,0.990751


F1 Macro: 0.9245260390942156
F1 Micro: 0.9849905384483055
F1 Weighted: 0.9849809342389966
{'eval_loss': 0.1006765142083168, 'eval_LOC': {'precision': 0.9301187980433263, 'recall': 0.9419674451521586, 'f1': 0.9360056258790436, 'number': 1413}, 'eval_MISC': {'precision': 0.8383594692400482, 'recall': 0.8559113300492611, 'f1': 0.8470444850700792, 'number': 812}, 'eval_ORG': {'precision': 0.894681960375391, 'recall': 0.8988999476165531, 'f1': 0.8967859942513717, 'number': 1909}, 'eval_PER': {'precision': 0.9667519181585678, 'recall': 0.950345694531741, 'f1': 0.9584786053882726, 'number': 1591}, 'eval_overall_precision': 0.9150121908742599, 'eval_overall_recall': 0.9177292576419214, 'eval_overall_f1': 0.9163687102119125, 'eval_overall_accuracy': 0.9849905384483055, 'eval_f1_micro': 0.9849905384483055, 'eval_f1_macro': 0.9245260390942156, 'eval_f1_weighted': 0.9849809342389966, 'eval_runtime': 39.2894, 'eval_samples_per_second': 87.225, 'eval_steps_per_second': 5.472, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
   # limpa buffers de IPC
 


 

In [ ]:
results = {}
trainer_all = {}
s = splits[6]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: std


Map: 100%|██████████| 3427/3427 [00:00<00:00, 21164.11 examples/s]
/tmp/ipykernel_7866/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.164000,0.041421,"{'precision': 0.9327837666455295, 'recall': 0.960809928151535, 'f1': 0.9465894465894468, 'number': 1531}","{'precision': 0.849146110056926, 'recall': 0.9104781281790437, 'f1': 0.8787432498772705, 'number': 983}","{'precision': 0.9293966623876765, 'recall': 0.8921749845964264, 'f1': 0.9104055328513047, 'number': 1623}","{'precision': 0.9668658337859859, 'recall': 0.9732094040459267, 'f1': 0.9700272479564033, 'number': 1829}",0.927695,0.937647,0.932644,0.989335,0.989335,0.936923,0.989380
2,0.014700,0.045679,"{'precision': 0.944015444015444, 'recall': 0.9581972566949706, 'f1': 0.9510534846029173, 'number': 1531}","{'precision': 0.8392523364485981, 'recall': 0.91353001017294, 'f1': 0.8748173404773502, 'number': 983}","{'precision': 0.9381443298969072, 'recall': 0.8971041281577325, 'f1': 0.9171653543307087, 'number': 1623}","{'precision': 0.9654427645788337, 'recall': 0.9775833788955713, 'f1': 0.9714751426242869, 'number': 1829}",0.930491,0.940161,0.935301,0.988848,0.988848,0.933645,0.988936
3,0.007100,0.046012,"{'precision': 0.9490651192778853, 'recall': 0.961463096015676, 'f1': 0.9552238805970149, 'number': 1531}","{'precision': 0.8677766895200784, 'recall': 0.901322482197355, 'f1': 0.8842315369261476, 'number': 983}","{'precision': 0.9290161892901619, 'recall': 0.9192852741836106, 'f1': 0.9241251161350263, 'number': 1623}","{'precision': 0.9747530186608123, 'recall': 0.9710224166211044, 'f1': 0.9728841413311422, 'number': 1829}",0.937667,0.943010,0.940331,0.990172,0.990172,0.942564,0.990158
4,0.003600,0.048619,"{'precision': 0.9506093649775497, 'recall': 0.9679947746570868, 'f1': 0.9592233009708737, 'number': 1531}","{'precision': 0.8825271470878578, 'recall': 0.9094608341810784, 'f1': 0.8957915831663327, 'number': 983}","{'precision': 0.9367167919799498, 'recall': 0.9211337030191005, 'f1': 0.9288598943771358, 'number': 1623}","{'precision': 0.9731800766283525, 'recall': 0.9721159103335156, 'f1': 0.9726477024070023, 'number': 1829}",0.942285,0.946866,0.944570,0.990834,0.990834,0.948880,0.990826
5,0.001100,0.051857,"{'precision': 0.9487179487179487, 'recall': 0.9666884389288047, 'f1': 0.9576188935619542, 'number': 1531}","{'precision': 0.8852621167161226, 'recall': 0.9104781281790437, 'f1': 0.8976930792377131, 'number': 983}","{'precision': 0.9371464487743557, 'recall': 0.9186691312384473, 'f1': 0.9278158058494089, 'number': 1623}","{'precision': 0.9763995609220637, 'recall': 0.9726626571897211, 'f1': 0.9745275267050124, 'number': 1829}",0.943349,0.946195,0.944770,0.990756,0.990756,0.947456,0.990751


F1 Macro: 0.9245260390942156
F1 Micro: 0.9849905384483055
F1 Weighted: 0.9849809342389966
{'eval_loss': 0.1006765142083168, 'eval_LOC': {'precision': 0.9301187980433263, 'recall': 0.9419674451521586, 'f1': 0.9360056258790436, 'number': 1413}, 'eval_MISC': {'precision': 0.8383594692400482, 'recall': 0.8559113300492611, 'f1': 0.8470444850700792, 'number': 812}, 'eval_ORG': {'precision': 0.894681960375391, 'recall': 0.8988999476165531, 'f1': 0.8967859942513717, 'number': 1909}, 'eval_PER': {'precision': 0.9667519181585678, 'recall': 0.950345694531741, 'f1': 0.9584786053882726, 'number': 1591}, 'eval_overall_precision': 0.9150121908742599, 'eval_overall_recall': 0.9177292576419214, 'eval_overall_f1': 0.9163687102119125, 'eval_overall_accuracy': 0.9849905384483055, 'eval_f1_micro': 0.9849905384483055, 'eval_f1_macro': 0.9245260390942156, 'eval_f1_weighted': 0.9849809342389966, 'eval_runtime': 36.2663, 'eval_samples_per_second': 94.495, 'eval_steps_per_second': 5.928, 'epoch': 5.0}




0

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
df = ner_collector.to_dataframes()

In [ ]:
df

(       split  eval_loss  overall_precision  overall_recall  overall_f1  \
 0        loc   0.054959           0.934578        0.939066    0.936817   
 1    reverse   0.193439           0.880701        0.864156    0.872350   
 2   semantic   0.082388           0.951739        0.954353    0.953044   
 3   heur_len   0.036307           0.960514        0.962907    0.961709   
 4  heur_rare   0.053442           0.944841        0.948884    0.946858   
 5        std   0.100677           0.915012        0.917729    0.916369   
 6        std   0.100677           0.915012        0.917729    0.916369   
 
    overall_accuracy  f1_micro  f1_macro  f1_weighted  runtime_s  \
 0          0.990728  0.990728  0.933049     0.990716    51.1556   
 1          0.974013  0.974013  0.891480     0.972969    66.7348   
 2          0.986872  0.986872  0.954010     0.986584    29.4755   
 3          0.992963  0.992963  0.962139     0.992926    62.6548   
 4          0.991022  0.991022  0.947829     0.990964    6

In [ ]:
nome_modelo = MODEL_NAME.split("/")[1]

In [ ]:
ner_collector.to_csv(base_name=nome_modelo)

('albert-base-v2_overall_metrics.csv', 'albert-base-v2_label_metrics.csv')